In [ ]:
%pip install yfinance

import yfinance as yf
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

In [ ]:
# reference spark session
spark = SparkSession.builder.getOrCreate()

In [ ]:
# extract stock data for Apple (AAPL) from Yahoo Finance
ticker = "AAPL"
df = yf.download(ticker, period="2y", interval="1d")
df = df.reset_index()
df.columns = [col.lower().replace(" ", "_") for col in df.columns]

In [ ]:
df["ticker"] = ticker
df["ingestion_time"] = pd.Timestamp.now()

In [ ]:
# convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

In [ ]:
# write to bronze data table
spark_df.write \
    .format("delta") \
    .mode("append") \
    .save("/path/to/bronze/aapl_stock_data")

print(f'Successfully ingested {ticker} stock data to bronze table.')